In [9]:
from dotenv import load_dotenv
import os
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from transformers import pipeline
import torch
from transformers import pipeline, AutoTokenizer



# Load API key
load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# Check key loaded
if not PINECONE_API_KEY:
    raise ValueError("❌ Pinecone API key not found in .env file.")

print(" Environment loaded.")


 Environment loaded.


In [10]:
# Load a single PDF file from the 'documents' folder
def load_pdf(path):
    loader = PyMuPDFLoader(path)
    documents = loader.load()
    print(f" Loaded {len(documents)} page(s).")
    return documents

# Load your file
doc_path = "documents/Article.pdf"
docs = load_pdf(doc_path)
docs


 Loaded 7 page(s).


[Document(metadata={'producer': 'Acrobat Distiller 15.0 (Macintosh)', 'creator': 'TeX', 'creationdate': '2017-01-21T20:03:26-08:00', 'source': 'documents/Article.pdf', 'file_path': 'documents/Article.pdf', 'total_pages': 7, 'format': 'PDF 1.4', 'title': 'SummaRuNNer: A Recurrent Neural Network Based Sequence Model for Extractive Summarization of Documents', 'author': 'Ramesh Nallapati, Feifei Zhai, Bowen Zhou', 'subject': 'Proceedings of the the Thirty-First AAAI Conference on Artificial Intelligence (AAAI-17)', 'keywords': 'Natural Language Processing and Knowledge Representation', 'moddate': '2017-01-22T21:56:57-08:00', 'trapped': '', 'modDate': "D:20170122215657-08'00'", 'creationDate': "D:20170121200326-08'00'", 'page': 0}, page_content='SummaRuNNer: A Recurrent Neural Network Based\nSequence Model for Extractive Summarization of Documents\nRamesh Nallapati, Feifei Zhai,∗Bowen Zhou\nnallapati@us.ibm.com, ffzhai2012@gmail.com, zhou@us.ibm.com\nIBM Watson\n1011 Kitchawan Road, Yorkto

In [11]:
# Split the PDF into smaller chunks
def chunk_documents(documents, chunk_size=800, chunk_overlap=50):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_documents(documents)
    print(f" Created {len(chunks)} chunks.")
    return chunks

# Split the loaded doc
chunks = chunk_documents(docs)
chunks


 Created 49 chunks.


[Document(metadata={'producer': 'Acrobat Distiller 15.0 (Macintosh)', 'creator': 'TeX', 'creationdate': '2017-01-21T20:03:26-08:00', 'source': 'documents/Article.pdf', 'file_path': 'documents/Article.pdf', 'total_pages': 7, 'format': 'PDF 1.4', 'title': 'SummaRuNNer: A Recurrent Neural Network Based Sequence Model for Extractive Summarization of Documents', 'author': 'Ramesh Nallapati, Feifei Zhai, Bowen Zhou', 'subject': 'Proceedings of the the Thirty-First AAAI Conference on Artificial Intelligence (AAAI-17)', 'keywords': 'Natural Language Processing and Knowledge Representation', 'moddate': '2017-01-22T21:56:57-08:00', 'trapped': '', 'modDate': "D:20170122215657-08'00'", 'creationDate': "D:20170121200326-08'00'", 'page': 0}, page_content='SummaRuNNer: A Recurrent Neural Network Based\nSequence Model for Extractive Summarization of Documents\nRamesh Nallapati, Feifei Zhai,∗Bowen Zhou\nnallapati@us.ibm.com, ffzhai2012@gmail.com, zhou@us.ibm.com\nIBM Watson\n1011 Kitchawan Road, Yorkto

In [12]:

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=0 if torch.cuda.is_available() else -1)

# Join text
texts = [doc.page_content for doc in chunks]
full_text = " ".join(texts)

# Tokenize and truncate to max 1024 tokens
inputs = tokenizer(full_text, return_tensors="pt", max_length=1024, truncation=True)


truncated_text = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)

# Summarize
summary = summarizer(truncated_text, max_length=512, min_length=100, do_sample=False)

# Show result
print("📝 Final Summary:\n")
print(summary[0]['summary_text'])


Device set to use cpu


📝 Final Summary:

SummaRuNNer is a Recurrent Neural Network BasedSequence Model for Extractive Summarization of Documents. We show that it achieves performance betterthan or comparable to state-of-the-art. It has the advantage of being very interpretable, since it allowsvisualization of its predictions broken up by abstract features such as information content, salience and novelty. We use the same neural networks used for summarizing CNN/DailyMail corpus. We also propose an attention-encoder-decoder for extractive single-document summariza-tion.


In [13]:
# Load the embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract just the text content from chunks
texts = [chunk.page_content for chunk in chunks]

# Generate embeddings (returns list of numpy arrays)
embeddings = model.encode(texts, show_progress_bar=True)

print(f" Generated {len(embeddings)} embeddings with dimension {len(embeddings[0])}")


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

 Generated 49 embeddings with dimension 384


In [14]:

# Connect to Pinecone with API key
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "book-summaries"

if pc.has_index(index_name):
    pc.delete_index(index_name)
    print(f" Deleted existing index '{index_name}'")

# Creating a new serverless index with 384-dimensional vectors
pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    vector_type="dense",
    spec=ServerlessSpec(cloud="aws", region="us-east-1") 
)
print(f" Created index '{index_name}' with dimension 384.")


 Deleted existing index 'book-summaries'
 Created index 'book-summaries' with dimension 384.


In [15]:
# Connect to the Pinecone index
index = pc.Index(index_name)

# Prepare the vectors and metadata for upsert
vectors = [
    {
        "id": f"chunk-{i}", 
        "values": embedding.tolist(), 
        "metadata": {"text": texts[i]}
    }
    for i, embedding in enumerate(embeddings)
]

# Upsert vectors into the namespace "book1"
namespace = "book1"
index.upsert(vectors=vectors, namespace=namespace)

print(f" Upserted {len(vectors)} vectors into namespace '{namespace}'.")


 Upserted 49 vectors into namespace 'book1'.


In [16]:
# Example question to search for
query = "Who are you?"

# Embed the query using the same model
query_embedding = model.encode(query).tolist()

# Query Pinecone to find the top 5 most similar vectors in the "book1" namespace
response = index.query(
    namespace="book1",
    vector=query_embedding, 
    top_k=5,
    include_values=False,
    include_metadata=True
)

# Retrieve and print the relevant chunks' text
retrieved_texts = [match["metadata"]["text"] for match in response["matches"]]

print(" Retrieved relevant chunks:")
for text in retrieved_texts:
    print("\n", text)


 Retrieved relevant chunks:


In [17]:

# Load the QA pipeline model from HuggingFace
qa_pipeline = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2",
    tokenizer="deepset/roberta-base-squad2"
)

# Example question
query = "Who are you?"

# Generate embedding for the question
query_embedding = model.encode(query).tolist()

# Query Pinecone 
response = index.query(
    namespace="book1",  
    vector=query_embedding, 
    top_k=5, 
    include_values=False,
    include_metadata=True 
)

# Combine the top 5 most relevant chunks
retrieved_texts = [match["metadata"]["text"] for match in response["matches"]]
context = " ".join(retrieved_texts)

answer = qa_pipeline(question=query, context=context)

# Output the answer
print("💬 Answer:", answer['answer'])


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Device set to use mps:0


💬 Answer: employee at IBM
